<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/generateText/NLP_DialoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small", torch_dtype=torch.float32)
tokenizer.pad_token = tokenizer.eos_token
tokenizer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


GPT2Tokenizer(name_or_path='microsoft/DialoGPT-small', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [6]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small",torch_dtype=torch.float32)
model

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [9]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [10]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [11]:
def combineText(example):
    return {
        "text": f"User: {example['context']} Bot: {example['response']}"
    }

In [12]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding="max_length", max_length=128)

In [13]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [14]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [15]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [16]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [17]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [18]:
trainSet = trainSet.map(combineText)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [19]:
testSet = testSet.map(combineText)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [20]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [21]:
testSet = testSet.map(encode, batched=True)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [22]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [23]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [24]:
print(set(trainSet[0]["labels"]))

{513, 514, 1043, 534, 2071, 25, 550, 1576, 2612, 1593, 72, 587, 588, 1101, 1613, 612, 616, 1641, 21608, 1139, 644, 3206, 651, 655, 18579, 1180, 674, 3241, 1194, 10416, 691, 12982, 41668, 714, 717, 1744, 2776, 220, 736, 749, 1265, 761, 262, 1290, 1808, 284, 287, 1312, 290, 812, 815, 3382, 1849, 318, 319, 326, 329, 339, 340, 2904, 345, 351, 867, 356, 355, 373, 2936, 892, 4988, 389, 393, 37264, 922, 11678, 1445, 422, 423, 32682, 1972, 3511, 1464, 1975, 1978, 1466, 2492, 18877, 460, 465, 466, 467, 468, 470, 475, 502, 503}


In [25]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='steps',
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=False,
    max_grad_norm = 1.0
)

In [26]:
trainSet = trainSet.select(range(1100))

In [27]:
testSet = testSet.select(range(670))

In [28]:
inputs = tokenizer.encode(
	input(">> User: ") + " Bot: " + tokenizer.eos_token, return_tensors='pt'
).to(model.device)

outputs = model.generate(input_ids=inputs, max_length=50, max_new_tokens=50,repetition_penalty=1.1,  do_sample=True, temperature=0.7, top_p=0.7, num_return_sequences=5)

>> User: I feel completely lost after my dog died. What should I do to cope day to day?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [29]:
tokenizer.decode(outputs, skip_special_tokens=True)

["I feel completely lost after my dog died. What should I do to cope day to day? Bot: We can't help you . We are all animals and we need comfort , not pain ! Dogma 101 by u jwcgj 42 bot is here if anyone else needs a hand with anything important today ... :D DDDd ? Thanks",
 "I feel completely lost after my dog died. What should I do to cope day to day? Bot: It's a good thing you're here . It can't get any worse , right ? ! :D DDDd ... You'll be fine though man .. Just keep going and don ' cry about it until tomorrow when we are all gone toooo",
 "I feel completely lost after my dog died. What should I do to cope day to day? Bot: Aww , that's so sweet . She was a beautiful girl ! Thanks for sharing this story with us all lt 3 It is really touching and heartwarming how much she means you too ... hugs :D D Edit spelling mistakes ? x3 s",
 "I feel completely lost after my dog died. What should I do to cope day to day? Bot: Aww , you're just a bot . Don't worry about it ! You'll be fine :D

In [30]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [31]:
trainer.evaluate(testSet)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'eval_loss': 8.094528198242188,
 'eval_model_preparation_time': 0.0018,
 'eval_runtime': 6.7448,
 'eval_samples_per_second': 99.336,
 'eval_steps_per_second': 6.227}

In [32]:
trainer.predict(testSet.select(range(80)))

PredictionOutput(predictions=array([[[ 25.190083,  20.328644,  19.92158 , ...,  23.659128,
          22.035858,  24.333473],
        [398.70972 , 360.67093 , 359.06384 , ..., 395.01428 ,
         398.91608 , 410.0853  ],
        [362.15216 , 326.86182 , 320.0437  , ..., 358.06866 ,
         361.63876 , 370.24478 ],
        ...,
        [419.74274 , 375.5734  , 373.1483  , ..., 414.201   ,
         419.924   , 433.83228 ],
        [420.5586  , 375.22424 , 372.34265 , ..., 412.80453 ,
         418.1659  , 433.92334 ],
        [425.9636  , 380.76953 , 377.83163 , ..., 421.52594 ,
         424.826   , 442.71872 ]],

       [[ 25.190083,  20.328644,  19.92158 , ...,  23.659128,
          22.035858,  24.333473],
        [398.70972 , 360.67093 , 359.06384 , ..., 395.01428 ,
         398.91608 , 410.0853  ],
        [362.15216 , 326.86182 , 320.0437  , ..., 358.06866 ,
         361.63876 , 370.24478 ],
        ...,
        [394.37717 , 349.94513 , 351.04865 , ..., 387.6764  ,
         392.1099

In [33]:
trainer.train()

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=207, training_loss=4.5105349038534115, metrics={'train_runtime': 130.4311, 'train_samples_per_second': 25.301, 'train_steps_per_second': 1.587, 'total_flos': 313387116134400.0, 'train_loss': 4.5105349038534115, 'epoch': 3.0})

In [34]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/mental-health-dialogpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
trainer.evaluate(testSet)

{'eval_loss': 3.3855626583099365,
 'eval_model_preparation_time': 0.0018,
 'eval_runtime': 7.3954,
 'eval_samples_per_second': 90.597,
 'eval_steps_per_second': 5.679,
 'epoch': 3.0}

In [36]:
trainer.predict(testSet.select(range(80)))

PredictionOutput(predictions=array([[[ 25.49744 ,  20.585709,  20.078077, ...,  23.11365 ,
          21.715027,  24.730785],
        [143.63153 , 134.07947 , 131.4706  , ..., 141.99893 ,
         143.26863 , 153.42424 ],
        [178.1252  , 167.42719 , 163.74484 , ..., 177.11588 ,
         184.19087 , 188.56958 ],
        ...,
        [100.10247 ,  96.0638  ,  93.253914, ...,  99.35485 ,
         107.92419 , 106.86241 ],
        [ 85.022934,  81.223076,  77.64591 , ...,  82.76014 ,
          87.59554 ,  90.368805],
        [227.26001 , 202.92079 , 201.36984 , ..., 217.51129 ,
         221.86655 , 247.11723 ]],

       [[ 25.49744 ,  20.585709,  20.078077, ...,  23.11365 ,
          21.715027,  24.730785],
        [143.63153 , 134.07947 , 131.4706  , ..., 141.99893 ,
         143.26863 , 153.42424 ],
        [178.1252  , 167.42719 , 163.74484 , ..., 177.11588 ,
         184.19087 , 188.56958 ],
        ...,
        [ 88.633606,  80.75555 ,  82.13293 , ...,  79.29682 ,
          82.9634